# GeoFire-X — Calibration scientifique B1/B2 (12 novembre 2026)
Inférence uniquement depuis les checkpoints achevés à l'époque 4. Aucun entraînement.

**Entrées :** les 11 partitions existantes de `dataset-sources.json`, et les sorties
enregistrées des notebooks contenant les deux checkpoints (Add Input → Notebook Output).
Choisir GPU T4 pour l'inférence et Internet désactivé. Aucun nouveau dataset à publier.

**Protocole figé :** 32 événements historiques explicitement lus dans les checkpoints,
13 759 tuiles ; seuil grossier 0,10–0,95 par 0,05, affiné ±0,05 par 0,005 ; sélection
Dice micro, égalité départagée par précision puis seuil supérieur. Les deux seuils sont
enregistrés avant toute inférence sur les 169 autres événements 2020. Tous leurs pixels
valides sont conservés. AP exacte et Brier restent calculés sur les probabilités originales.
Les 32 événements ayant déjà servi à la validation de l'entraînement, ils ne constituent
pas un test indépendant. Aucun pixel 2021 n'est ouvert.

Le code embarqué ne dépend ni du dépôt, ni d'Internet, ni de Spark/rasterio/pyarrow.
L'audit des pixels train est une lecture CPU sans prédiction ni apprentissage.


In [ ]:
import base64, hashlib, importlib.util, io, json, os, sys, tempfile, time, zipfile
from pathlib import Path
missing = [name for name in ("numpy", "torch", "matplotlib") if importlib.util.find_spec(name) is None]
assert not missing, f"Dépendances absentes : {missing}. Voir requirements-calibration.txt."
import numpy as np, torch, matplotlib
print({"python": sys.version, "numpy": np.__version__, "torch": torch.__version__, "matplotlib": matplotlib.__version__})
DEVICE = "cuda"  # mettre "cpu" explicitement pour une exécution sans accélérateur
BATCH_SIZE, WORKERS, CACHE_MB = 32, 4, 64
CHECKPOINTS = {
    "B1": Path("/kaggle/working/B1-continued-20261112_135150/checkpoint_e005_u0023404_epoch_complete.pt"),
    "B2": Path("/kaggle/working/B2-continued-20261112_141330/checkpoint_e005_u0023404_epoch_complete.pt"),
}
# Si nouvelle session : définir ces chemins sous /kaggle/input/... après avoir
# attaché les sorties sauvegardées des notebooks B1/B2. Aucun upload de dataset.
OUT = Path("/kaggle/working") / ("calibration-B1-B2-" + time.strftime("%Y%m%d_%H%M%S"))


In [ ]:
# Extraction du code seulement (jamais des archives du corpus).
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '659ccf0e07c494c3a4c93d527af13054e68019f74bd71c8c86458601b32fa61f'
workspace = Path(tempfile.mkdtemp(prefix="geofirex-calibration-code-", dir="/tmp"))
with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
    bundle.extractall(workspace)
sys.path.insert(0, str(workspace))
os.environ["PYTHONPATH"] = str(workspace) + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Code autonome vérifié :", workspace)


## Vérification des entrées
Les deux fichiers portent le même nom : l'identité B1/B2 est vérifiée à partir du
champ `config.model`, jamais déduite du nom seul. Le chemin explicite est prioritaire.
En nouvelle session, une recherche dans les sorties montées est possible si le
fichier demandé n'existe plus dans `/kaggle/working`. Une ambiguïté provoque un arrêt.


In [ ]:
filename = "checkpoint_e005_u0023404_epoch_complete.pt"
for name, path in list(CHECKPOINTS.items()):
    if not path.is_file():
        candidates = list(Path("/kaggle/input").rglob(filename))
        matching = []
        for candidate in candidates:
            # Uniquement vos checkpoints de confiance, attachés comme sorties.
            state = torch.load(candidate, map_location="cpu", weights_only=False)
            if state.get("config", {}).get("model") == name:
                matching.append(candidate)
            del state
        assert len(matching) == 1, f"{name}: spécifier le chemin exact CHECKPOINTS ; candidats={matching}"
        CHECKPOINTS[name] = matching[0]
assert CHECKPOINTS["B1"].resolve() != CHECKPOINTS["B2"].resolve()
assert DEVICE == "cpu" or torch.cuda.is_available(), "Sélectionner un accélérateur GPU ou DEVICE=cpu"
print("Checkpoints :", CHECKPOINTS)


In [ ]:
from geofirex.calibration_mounts import restore_verified
expected = ['ed88f93ae8329c57f34709fdf3a59e3ec3272db8238f843d4329f2cb23e293d0', '4aa8b2818192a2cc67941f1c1bbb8a1001e144114486bb05ecaa9b504993c6fa', 'b5caf48618be26a75a480d91401f9de0d0936607936d09556f6308c1d76c5dcd', 'ba72fbbc58f398a7bcd471d5cb21aebe4d11ed68da02caa912c2e746ccadc04d', 'ada2f8b9a3f7f340efed521b15bf3fe4e3b4151fa39eab06ca2055518f58ab58', '3bcde37b262d5828f98153350d961e0ad771032fdedb04ff180ace536e4cb33a', '001a171f309ac58f4160269d46b9163b0c0c26c7a488cc545f6fa1b42e55d72a', '906b02993f756f4b6098ac8da11ecac173f59d20ebda70c8d2bd3bb37b8f6dbd', '793223fda8a5f0d06e792bcd09b2604a97252952a5323b144184a8e4f1496a4f', 'af3015768e0174071c4d197ad31681ef20fbe0cc1de2c1084254bfef5b91393f', '49e5551860a6d14c020c86dc33118a7cd2e0ca3dc9b470e6e95af6061817fa81']
maps = list(Path("/kaggle/input").rglob("mount-map.json"))
# Filtrer seulement les 11 identités attendues ; les sorties notebook peuvent
# aussi contenir des anciennes copies de fichiers de contrôle.
maps = [p for p in maps if json.loads(p.read_text())["identity"] in expected]
corpus = restore_verified(maps, expected,
    Path(tempfile.mkdtemp(prefix="geofirex-calibration-mount-", dir="/tmp")) / "corpus")
index = corpus / "index"
print("Corpus relié sans reconstruction :", corpus)


## Calcul complet
Exécuter cette cellule une seule fois. Elle réalise l'audit train CPU, l'inférence
des 32 événements pour B1/B2, le verrouillage des seuils, puis les 169 événements
aux seuls seuils 0,5 et fixés. Les tris/fusions AP exacte utilisent le CPU et le disque,
et peuvent prendre plus de temps que l'inférence. Aucun débit GPU ou délai n'est promis.
Scratch : prévoir environ 12–15 Go libres pour la phase de validation la plus grande,
hors sorties déjà présentes ; les runs AP temporaires sont supprimés après calcul.
Les journaux indiquent l'événement terminé. Une interruption ne produit pas de
`status_complete.json` et ne permet pas de présenter la validation comme complète.


In [ ]:
import subprocess
command = [sys.executable, str(workspace / "calibrate_compact.py"),
    "--corpus", str(corpus), "--index", str(index),
    "--b1", str(CHECKPOINTS["B1"]), "--b2", str(CHECKPOINTS["B2"]),
    "--out", str(OUT), "--device", DEVICE, "--batch-size", str(BATCH_SIZE),
    "--workers", str(WORKERS), "--cache-mb", str(CACHE_MB)]
print("Commande :", command, flush=True)
subprocess.run(command, check=True, env=os.environ.copy())
assert json.loads((OUT / "status_complete.json").read_text())["status"] == "COMPLETE"


## Comparaison et visualisations
Les lignes baseline/calibré ont rigoureusement les mêmes AP et Brier. Les CSV
contiennent TP/FP/FN/TN, Dice/IoU/précision/rappel, couverture et prévalence. Les
distributions macro donnent le même poids à chaque événement ; les métriques micro
agrègent les pixels. Les sous-domaines feu nouveau / déjà observé décomposent les erreurs.


In [ ]:
from IPython.display import display, Image
comparison = json.loads((OUT / "comparison.json").read_text())
try:
    import pandas as pd
    display(pd.DataFrame(comparison["rows"]))
except ImportError:
    print(json.dumps(comparison["rows"], indent=2))
print("Seuils fixés :", json.loads((OUT / "thresholds_locked.json").read_text())["thresholds"])
for name in ("calibration_curves.png", "calibration32_diagnostics.png", "holdout169_diagnostics.png"):
    display(Image(filename=str(OUT / name)))
manifest = json.loads((OUT / "app_manifest.json").read_text())
for case in manifest["maps"]:
    display(Image(filename=str(OUT / case["directory"] / case["preview"])))


## Export pour React/FastAPI et conservation
Télécharger le dossier `calibration-B1-B2-*` depuis les outputs du notebook après
Save Version. `app_manifest.json` décrit les seuils, scores, distributions, fichiers
NPZ et couches PNG transparentes à résolution native. Erreurs : TN=0, TP=1 (vert),
FP=2 (rouge), FN=3 (bleu), invalide=255. `valid` est toujours fourni séparément.
Ces mosaïques sont des grilles de pixels : pas de géoréférencement inventé, pas de
GeoTIFF sans CRS/affine certifiés. L'application de production doit vérifier le SHA
du checkpoint et le normaliseur associés au seuil avant de l'utiliser. Le serveur
historique ne charge pas directement ce format de checkpoint compact.

Les 169 événements ne doivent pas servir à réajuster le seuil. Voir `DIAGNOSTIC.md`
pour les deux expériences futures proposées, qui nécessitent une autorisation GPU.
